# Practice Lab: Function Calling Deep Dive (Lesson 10.1)

Module 10 · 8 exercises · Three providers + tool choice + validation + tokens + parallel + security

Exercises 1-5 run locally (pure Python).


# Lesson 10.1: From Text to Action — Function Calling Deep Dive

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 4, 5 run **locally** (pure Python). Ex 6, 7, 8 are design.


In [ ]:
import json
import numpy as np
import re
import time
np.random.seed(42)


---
## Exercise 1 (Easy): Three-Provider Schema Comparison


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

openai_tool = {"type":"function","function":{"name":"get_weather",
    "description":"Get current weather for a city",
    "parameters":{"type":"object","properties":{
        "city":{"type":"string","description":"City name"},
        "unit":{"type":"string","enum":["celsius","fahrenheit"],"default":"celsius"}},
    "required":["city"]}}}

anthropic_tool = {"name":"get_weather","description":"Get current weather for a city",
    "input_schema":{"type":"object","properties":{
        "city":{"type":"string","description":"City name"},
        "unit":{"type":"string","enum":["celsius","fahrenheit"]}},
    "required":["city"]}}

comparison = [
    ("Schema format", "Python func+docstring", "JSON in 'parameters'", "JSON in 'input_schema'"),
    ("Tool wrapper", "tools=[func]", "tools=[{type:'function',...}]", "tools=[{name:...,input_schema}]"),
    ("Auto-schema", "YES (type hints)", "No (manual)", "No (manual)"),
    ("Response", "function_call parts", "tool_calls[]", "content[type=tool_use]"),
    ("Result", "FunctionResponse", "role='tool' msg", "tool_result block"),
    ("Parallel", "Multi-part", "Multiple tool_calls", "Multiple tool_use"),
]

print("Three-Provider Schema Comparison:")
print(f"{'Aspect':<16} {'Gemini':<24} {'OpenAI':<26} {'Anthropic'}")
print("-"*80)
for a, g, o, an in comparison:
    print(f"  {a:<14} {g:<24} {o:<26} {an}")

print(f"\nGemini=simplest (auto-schema), OpenAI=most mature, Anthropic=cleanest")


</details>


---
## Exercise 2 (Easy): Tool Choice Modes


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

def sim_choice(query, mode, tools):
    relevant = any(kw in query.lower() for t in tools for kw in t["kw"])
    if mode == "none": return "text_only", None
    elif mode == "auto":
        if relevant:
            m = [t for t in tools if any(k in query.lower() for k in t["kw"])]
            return "tool_call", m[0]["name"]
        return "text_only", None
    elif mode == "required":
        m = [t for t in tools if any(k in query.lower() for k in t["kw"])]
        return "tool_call", (m[0]["name"] if m else tools[0]["name"])
    return "tool_call", "get_weather"

tools = [{"name":"get_weather","kw":["weather","temp"]},
         {"name":"calculate","kw":["calculate","math"]},
         {"name":"search","kw":["search","find"]}]
queries = ["What's the weather in Hyderabad?", "Tell me a joke", "Calculate 14999*1.18"]

print("Tool Choice Modes:")
for q in queries:
    print(f"\n  '{q}':")
    for mode in ["auto","required","forced","none"]:
        action, tool = sim_choice(q, mode, tools)
        print(f"    {mode:<10} -> {action:<12} tool={tool or 'none'}")

print(f"\nCross-Provider: auto/AUTO/auto, none/NONE/none, required/ANY/any")
print(f"Parallel: ON by default. Max tools: 128. Best quality: 10-20.")


</details>


---
## Exercise 3 (Medium): Strict Mode + Pydantic Validation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class WeatherArgs:
    VALID_UNITS = ["celsius", "fahrenheit"]
    def __init__(self, city, unit="celsius"):
        errors = []
        if not isinstance(city, str) or len(city) < 2:
            errors.append("city must be string with 2+ chars")
        if unit not in self.VALID_UNITS:
            errors.append(f"unit must be one of {self.VALID_UNITS}")
        if errors: raise ValueError("; ".join(errors))
        self.city = city; self.unit = unit

def validate_and_execute(name, raw_args, attempt=1, max_attempts=3):
    try:
        if name == "get_weather":
            v = WeatherArgs(**raw_args)
            return {"success":True, "result":{"city":v.city,"temp":34,"unit":v.unit}}
        return {"success":False, "error":f"Unknown tool: {name}"}
    except (ValueError, TypeError) as e:
        if attempt < max_attempts:
            return {"success":False, "error":str(e), "action":"RETRY", "attempt":attempt}
        return {"success":False, "error":f"Failed after {max_attempts}: {e}", "action":"ABORT"}

print("Strict Mode + Pydantic Validation:")
tests = [("Valid",{"city":"Hyderabad","unit":"celsius"}),
         ("Bad unit",{"city":"Mumbai","unit":"kelvin"}),
         ("Missing city",{"unit":"celsius"}),
         ("Empty city",{"city":"","unit":"celsius"}),
         ("Wrong type",{"city":123,"unit":"celsius"})]

for label, args in tests:
    r = validate_and_execute("get_weather", args)
    s = "PASS" if r["success"] else "FAIL"
    print(f"  [{s}] {label}: {args}")
    if not r["success"]: print(f"       Error: {r['error']}")

print(f"\nNEVER silently fallback to empty dict")
print(f"Correct: strict mode + Pydantic + 3 retries + graceful error")


</details>


---
## Exercise 4 (Medium): Token Cost Calculator


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
TPT = 125
PRICES = {"gpt-4o":2.50,"gpt-4o-mini":0.15,"claude-sonnet":3.00,"gemini-flash":0.10}

print("Token Cost Calculator:")
print(f"\n1. Overhead by Tool Count:")
print(f"   {'Tools':<8} {'Tokens':>8} {'Per 1K req':>12} {'10-turn':>10}")
for n in [5,10,20,50]:
    t = n*TPT
    print(f"   {n:<8} {t:>8,} {t*1000:>12,} {t*10:>10,}")

t20 = 20*TPT
print(f"\n2. Cost for 1K Requests (20 tools={t20:,} tok/req):")
print(f"   {'Model':<16} {'No Cache':>10} {'Cached':>10} {'Savings':>10}")
for m, p in PRICES.items():
    nc = t20*1000*p/1e6; c = nc*0.1; s = nc-c
    print(f"   {m:<16} ${nc:>8.2f} ${c:>8.2f} ${s:>8.2f}")

print(f"\n3. Multi-turn (20 tools, 10 turns):")
for m, p in PRICES.items():
    print(f"   {m:<16}: ${t20*10*p/1e6:.4f}/conversation")

print(f"\n4. 128K Budget: system 5%(6.4K), tools 15%(19.2K), conv 30%, RAG 45%, buffer 5%")
print(f"Prompt caching saves 90%. Keep tool arrays STABLE for cache hits.")


</details>


---
## Exercise 5 (Medium): Parallel Function Calling


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time

def sim_tool(name, lat_ms):
    time.sleep(lat_ms/1000)
    results = {"get_weather":{"temp":34},"search_flights":{"cheapest":"6500 INR"},"search_hotels":{"cheapest":"2800/night"}}
    return results.get(name, {})

tools = [("get_weather",200),("search_flights",500),("search_hotels",400)]

print("Parallel vs Sequential:")
start = time.time()
for name, lat in tools: sim_tool(name, lat)
seq = (time.time()-start)*1000
par = max(t[1] for t in tools)

print(f"  Sequential: {seq:.0f}ms")
print(f"  Parallel:   {par}ms (limited by slowest)")
print(f"  Speedup:    {seq/par:.1f}x!")

print(f"\nProvider Patterns:")
for p, how in [("OpenAI","Multiple tool_calls -> append all -> re-call once"),
               ("Gemini","Multi-part -> send all FunctionResponse together"),
               ("Claude","Multiple tool_use -> send all tool_result in one msg")]:
    print(f"  {p}: {how}")
print(f"Fails: o-series reasoning models, dependent tools")


</details>


---
## Exercise 6 (Challenge): Dynamic Tool Loading
Design/architecture. See practice-lab-10_1.html.


In [ ]:
# Exercise 6: Dynamic Tool Loading
pass


---
## Exercise 7 (Challenge): Security Guardrails
Design/architecture. See practice-lab-10_1.html.


In [ ]:
# Exercise 7: Security Guardrails
pass


---
## Exercise 8 (Challenge): Indic Voice Agent
Design/architecture. See practice-lab-10_1.html.


In [ ]:
# Exercise 8: Indic Voice Agent
pass
